# Residual Hybrid Digital Twin — Methanol Reactor

**Approach:** Physics baseline + ML residual correction (**Option A**).

```text
Plant inputs u ──► Physics ODE (η) ──► y_phys
                         │
                         ▼
              train: r = y_excel − y_phys
                         │
Plant inputs u ──► ML residual model ──► r̂ = f_ML(u)
                         │
                         ▼
                   ŷ = y_phys + r̂   → clip
```

| Role | Variables |
|------|-----------|
| **Inputs (4)** | Inlet T (°C), Inlet P (bar_g), Heating media T (°C), Methanol flow (kgmole/h) |
| **Outputs (8)** | Temp_0.2m … Temp_1m (°C), P_out (bar_g), Q (kW), Conversion (%) |

**Reaction:** CH₃OH → CO + 2H₂  
**Data:** `Methanol Reforming Data.xlsx`  
**ML Option A:** residual model sees **plant inputs `u` only** (not `y_phys`).


## 1. Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor

DATA_FILE = Path("Methanol Reforming Data.xlsx")
# T_heat_C is a per-case plant input from Methanol Reforming Data.xlsx
MEASUREMENT_POSITIONS_M = np.array([0.2, 0.4, 0.6, 0.8, 1.0])
P_ATM_BAR = 1.01325

INPUT_COLS = ["T_in_C", "P_in_bar_g", "T_heat_C", "F_in_kgmole_h"]
OUTPUT_COLS = [
    "T_0.2m_C", "T_0.4m_C", "T_0.6m_C", "T_0.8m_C", "T_1m_C",
    "P_out_bar_g",
    "Q_total_kW",
    "X_overall_pct",
]
TEMP_COLS = ["T_0.2m_C", "T_0.4m_C", "T_0.6m_C", "T_0.8m_C", "T_1m_C"]

print("Libraries loaded.")
print("Data file:", DATA_FILE.resolve())


## 2. Load and clean `Methanol Reforming Data.xlsx`


In [ ]:
def load_data(path: Path) -> pd.DataFrame:
    # Load operating cases from Methanol Reforming Data.xlsx (Sheet1).
    raw = pd.read_excel(path, sheet_name=0, header=0, skiprows=[1])  # row1 = units
    n_raw = len(raw)

    data = raw.rename(
        columns={
            "Inlet Temperature (Ti)": "T_in_C",
            "Inlet Pressure (Pi)": "P_in_bar_g",
            "Heating media Temp (Th)": "T_heat_C",
            "Inlet molar flow rate (mi)": "F_in_kgmole_h",
            "Temperature at 0.2m (T1)": "T_0.2m_C",
            "Temperature at 0.4m (T2)": "T_0.4m_C",
            "Temperature at 0.6m (T3)": "T_0.6m_C",
            "Temperature at 0.8m (T4)": "T_0.8m_C",
            "Temperature at 1m (T5)": "T_1m_C",
            "Pressure at 1m (P5)": "P_out_bar_g",
        }
    )

    # Excel segment heat flows are negative for heat into the reactor → flip and sum.
    # Ensure numeric dtypes after dropping the Excel units row
    for _col in [
        "T_in_C", "P_in_bar_g", "T_heat_C", "F_in_kgmole_h",
        "T_0.2m_C", "T_0.4m_C", "T_0.6m_C", "T_0.8m_C", "T_1m_C", "P_out_bar_g",
        "Conversion at 1m (X-5)",
    ]:
        if _col in data.columns:
            data[_col] = pd.to_numeric(data[_col], errors="coerce")
    for _col in [c for c in data.columns if str(c).startswith("Heat flow")]:
        data[_col] = pd.to_numeric(data[_col], errors="coerce")

    heat_cols = [c for c in data.columns if str(c).startswith("Heat flow")]
    data["Q_total_kW"] = -data[heat_cols].sum(axis=1)

    # Excel stores conversion as a fraction → percent.
    data["X_overall_pct"] = data["Conversion at 1m (X-5)"] * 100.0

    print(f"Total rows in file          : {n_raw}")

    ok = (
        data["X_overall_pct"].between(0, 100)
        & data["Q_total_kW"].between(-50, 50)
        & data[["T_in_C", "P_in_bar_g", "T_heat_C", "F_in_kgmole_h"]].notna().all(axis=1)
    )
    cleaned = data.loc[ok].reset_index(drop=True)
    print(f"Valid rows                  : {len(cleaned)}")
    return cleaned
df = load_data(DATA_FILE)
display(df[INPUT_COLS + OUTPUT_COLS].head())


## 3. Fixed reactor physics parameters

Thermo / geometry / kinetics used by the physics baseline ODE.

In [ ]:
R_kcal = 1.987
delta_H_ref_kcal = 2.2e4
T_ref_C = 25.0

KCAL_PER_KGMOLE_TO_J_PER_MOL = 4184.0 / 1000.0
R = R_kcal * KCAL_PER_KGMOLE_TO_J_PER_MOL
delta_H_ref = delta_H_ref_kcal * KCAL_PER_KGMOLE_TO_J_PER_MOL

D = 0.1
L = 1.0
h_o = 115.0
# T_heat_C is a per-case plant input
epsilon = 0.33
dp = 1.0e-3
Ac = np.pi * D**2 / 4
perimeter = np.pi * D

MW_CH3OH = 32.04e-3
MW_CO = 28.01e-3
MW_H2 = 2.016e-3

Cp_CH3OH = 43.0
Cp_CO = 29.0
Cp_H2 = 29.0

delta_Cp = Cp_CO + 2.0 * Cp_H2 - Cp_CH3OH

A_kinetic = 7.0e10
E_kinetic = 19120.0
A_CO = 1.0e7
E_CO = 6036.8

ETA = 1.0



mu_gas = 2.0e-5          # Gas viscosity [Pa·s]
k_gas = 0.035            # Gas thermal conductivity [W/(m·K)]


def bed_side_htc(F_in_mol_s):
    # Specchia bed-side wall coefficient h_i [W/(m²·°C)]
    mass_flux = F_in_mol_s * MW_CH3OH / Ac
    Re_p = dp * mass_flux / mu_gas
    Nu_w = 2.0 * epsilon + 0.0835 * (Re_p ** 0.91)
    return Nu_w * k_gas / dp


def overall_U(F_in_mol_s):
    # Overall wall coefficient. h_heat = U, from the inlet methanol flow.
    h_i = bed_side_htc(F_in_mol_s)
    return 1.0 / (1.0 / h_o + 1.0 / h_i)

def reaction_enthalpy(T_C):
    return delta_H_ref + delta_Cp * (float(T_C) - T_ref_C)


print(f"R = {R_kcal} kcal/(kgmole·K)  ->  {R:.6f} J/(mol·K)")
print(f"ΔH(25°C) = {delta_H_ref_kcal:.3e} kcal/kgmole  ->  {delta_H_ref:.3e} J/mol")
print(f"ΔCp = {delta_Cp:.2f} J/(mol·K)")
print(f"ΔH(200°C) = {reaction_enthalpy(200)/4184*1000:.3e} kcal/kgmole")
print(f"D={D} m, L={L} m, T_heat=per-case input")


## 4. Physics ODE baseline

First-principles PFR model. This is the **baseline** prediction that ML will correct.

In [ ]:
def bar_g_to_Pa(P_bar_g):
    return (P_bar_g + P_ATM_BAR) * 1.0e5


def Pa_to_bar_g(P_Pa):
    return P_Pa / 1.0e5 - P_ATM_BAR


def kgmole_h_to_mol_s(F_kgmole_h):
    return F_kgmole_h * 1000.0 / 3600.0


def reaction_rate(T_C, P, F_CH3OH, F_CO, F_H2, eta=None):
    if eta is None:
        eta = ETA
    T_abs = T_C + 273.15
    F_total = max(F_CH3OH + F_CO + F_H2, 1e-12)
    C_total = P / (R * T_abs) / 1000.0  # kgmole/m³

    C_CH3OH = (F_CH3OH / F_total) * C_total
    C_CO = (F_CO / F_total) * C_total
    num = A_kinetic * np.exp(-E_kinetic / (R_kcal * T_abs)) * C_CH3OH
    den = 1.0 + A_CO * np.exp(-E_CO / (R_kcal * T_abs)) * C_CO
    return eta * (num / den) * 1000.0  # mol/(m³·s)


def reactor_model(z, y, h_heat, T_heat_C, eta=None):
    if eta is None:
        eta = ETA
    F_CH3OH = max(y[0], 1e-12)
    F_CO = max(y[1], 0.0)
    F_H2 = max(y[2], 0.0)
    T_C = y[3]
    P = max(y[4], 1.0e3)

    r = reaction_rate(T_C, P, F_CH3OH, F_CO, F_H2, eta=eta)
    rpl = r * Ac

    q_transfer = h_heat * perimeter * (T_heat_C - T_C)
    Cp_flow = max(F_CH3OH * Cp_CH3OH + F_CO * Cp_CO + F_H2 * Cp_H2, 1e-9)
    delta_H_T = reaction_enthalpy(T_C)
    dT_dz = (q_transfer - delta_H_T * rpl) / Cp_flow

    F_total = F_CH3OH + F_CO + F_H2
    mass_flow = F_CH3OH * MW_CH3OH + F_CO * MW_CO + F_H2 * MW_H2
    MW_mix = mass_flow / max(F_total, 1e-12)
    T_abs = T_C + 273.15
    rho = P * MW_mix / (R * T_abs)
    velocity = mass_flow / max(rho * Ac, 1e-12)
    mu = mu_gas

    viscous = 150.0 * mu * (1.0 - epsilon) ** 2 / (epsilon ** 3 * dp ** 2) * velocity
    inertial = 1.75 * rho * (1.0 - epsilon) / (epsilon ** 3 * dp) * velocity ** 2
    dP_dz = -(viscous + inertial)

    return [-rpl, rpl, 2.0 * rpl, dT_dz, dP_dz, q_transfer]


def predict_physics(T_in_C, P_in_bar_g, T_heat_C, F_in_kgmole_h, eta=None, n_points=80):
    if eta is None:
        eta = ETA
    F_in = kgmole_h_to_mol_s(F_in_kgmole_h)
    P_in_Pa = bar_g_to_Pa(P_in_bar_g)
    h_heat = overall_U(F_in)  # revised physics: h_heat = U from inlet flow
    y0 = [F_in, 0.0, 0.0, float(T_in_C), P_in_Pa, 0.0]
    z_eval = np.linspace(0.0, L, n_points)

    sol = solve_ivp(
        lambda z, y: reactor_model(z, y, h_heat, float(T_heat_C), eta=eta),
        (0.0, L),
        y0,
        t_eval=z_eval,
        method="BDF",
        rtol=1e-5,
        atol=1e-8,
    )
    if not sol.success:
        raise RuntimeError(sol.message)

    F_out = float(sol.y[0, -1])
    T_profile = sol.y[3]
    T_axial = np.interp(MEASUREMENT_POSITIONS_M, sol.t, T_profile)
    X_pct = (F_in - F_out) / F_in * 100.0

    return {
        "T_0.2m_C": float(T_axial[0]),
        "T_0.4m_C": float(T_axial[1]),
        "T_0.6m_C": float(T_axial[2]),
        "T_0.8m_C": float(T_axial[3]),
        "T_1m_C": float(T_axial[4]),
        "P_out_bar_g": float(Pa_to_bar_g(sol.y[4, -1])),
        "Q_total_kW": float(sol.y[5, -1] / 1000.0),
        "X_overall_pct": float(X_pct),
        "z_m": sol.t,
        "T_profile_C": T_profile,
    }


def physics_vector(T_in_C, P_in_bar_g, T_heat_C, F_in_kgmole_h, eta=None):
    p = predict_physics(T_in_C, P_in_bar_g, T_heat_C, F_in_kgmole_h, eta=eta)
    return np.array([p[c] for c in OUTPUT_COLS], dtype=float)


print("Physics baseline ready.")


## 5. Train / test split

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train rows: {len(train_df)}")
print(f"Test rows : {len(test_df)}")

## 6. Calibrate physics rate scale η

Fit η so physics conversion matches Excel on a train sample **before** residual learning.

In [ ]:
def calibrate_eta(df_train, n_cases=40):
    sample = df_train.sample(n=min(n_cases, len(df_train)), random_state=42)
    first_error = None

    def objective(log_eta):
        nonlocal first_error
        eta = 10 ** log_eta
        errs = []
        for _, row in sample.iterrows():
            try:
                pred = physics_vector(
                    row["T_in_C"], row["P_in_bar_g"], row["T_heat_C"], row["F_in_kgmole_h"], eta=eta
                )
                errs.append((pred[-1] - row["X_overall_pct"]) ** 2)
            except Exception as exc:
                if first_error is None:
                    first_error = exc
                errs.append(1e6)
        return np.mean(errs)

    print("Calibrating eta...")
    result = minimize_scalar(
        objective, bounds=(-2.0, 2.0), method="bounded", options={"xatol": 0.05}
    )
    if first_error is not None and result.fun >= 1.0e6:
        raise RuntimeError(
            f"Eta calibration failed for all sample cases. First physics error: {first_error!r}."
        )
    return float(10 ** result.x), float(result.fun)


ETA, eta_obj = calibrate_eta(train_df, n_cases=40)
print(f"Calibrated ETA = {ETA:.6e}  (sample conversion MSE = {eta_obj:.4f})")


## 7. Run physics baseline on train and test

Build `Y_phys` and residual targets `R = Y_excel − Y_phys` (only where physics succeeds).

In [ ]:
def run_physics_matrix(data, eta, label="set"):
    rows = []
    n_fail = 0
    first_error = None

    for _, row in data.iterrows():
        try:
            rows.append(
                physics_vector(
                    row["T_in_C"], row["P_in_bar_g"], row["T_heat_C"], row["F_in_kgmole_h"], eta=eta
                )
            )
        except Exception as exc:
            n_fail += 1
            if first_error is None:
                first_error = exc
            rows.append(np.full(len(OUTPUT_COLS), np.nan))

        if (len(rows) % 100) == 0:
            print(f"  {label}: {len(rows)}/{len(data)}")

    if first_error is not None:
        print(f"  WARNING [{label}]: {n_fail} physics failures. First error: {first_error!r}")

    return np.vstack(rows)


print("Physics on train...")
Y_phys_train = run_physics_matrix(train_df, ETA, label="train")
print("Physics on test...")
Y_phys_test = run_physics_matrix(test_df, ETA, label="test")

U_train = train_df[INPUT_COLS].to_numpy(dtype=float)
U_test = test_df[INPUT_COLS].to_numpy(dtype=float)
Y_excel_train = train_df[OUTPUT_COLS].to_numpy(dtype=float)
Y_excel_test = test_df[OUTPUT_COLS].to_numpy(dtype=float)

ok_train = np.isfinite(Y_phys_train).all(axis=1)
ok_test = np.isfinite(Y_phys_test).all(axis=1)

print(
    f"Finite physics — train {ok_train.sum()}/{len(ok_train)}, "
    f"test {ok_test.sum()}/{len(ok_test)}"
)

if ok_train.sum() == 0:
    raise RuntimeError("All physics predictions failed. Re-run Steps 4–6 first.")

# Residual targets (Option A training labels)
R_train = Y_excel_train - Y_phys_train
R_test = Y_excel_test - Y_phys_test

print("Residual stats on finite train rows:")
resid_stats = pd.DataFrame(
    {
        "Output": OUTPUT_COLS,
        "mean_r": np.nanmean(R_train[ok_train], axis=0),
        "mae_r": np.nanmean(np.abs(R_train[ok_train]), axis=0),
    }
)
display(resid_stats.round(4))

## 8. ML residual model — Option A (`u` → residual)

Black-box learns **physics error only**:

```text
r̂ = f_ML(u)     where u = [T_in, P_in, T_heat, F_in]
```

ML does **not** see `y_phys`. Trained only on rows where physics succeeded.

In [ ]:
U_tr = U_train[ok_train]
R_tr = R_train[ok_train]

residual_model = MultiOutputRegressor(
    HistGradientBoostingRegressor(
        max_depth=6,
        learning_rate=0.08,
        max_iter=250,
        l2_regularization=0.1,
        random_state=42,
    )
)
residual_model.fit(U_tr, R_tr)
print("Residual ML trained: u → (y_excel − y_phys).")

# Predict residuals for full train/test matrices
R_hat_train = residual_model.predict(U_train)
R_hat_test = residual_model.predict(U_test)
print(f"R_hat_train shape: {R_hat_train.shape}, R_hat_test shape: {R_hat_test.shape}")

# Feature importance on residual learners
fi_rows = []
for i, est in enumerate(residual_model.estimators_):
    pi = permutation_importance(
        est,
        U_tr,
        R_tr[:, i],
        n_repeats=10,
        random_state=42,
        n_jobs=-1,
    )
    fi_rows.append(np.maximum(pi.importances_mean, 0.0))

fi_matrix = np.vstack(fi_rows)
importance_df = pd.DataFrame(fi_matrix, columns=INPUT_COLS, index=OUTPUT_COLS)
importance_mean = (
    pd.DataFrame({"feature": INPUT_COLS, "importance": fi_matrix.mean(axis=0)})
    .sort_values("importance", ascending=True)
    .reset_index(drop=True)
)

print("Mean permutation importance across 8 residual outputs:")
display(importance_mean.sort_values("importance", ascending=False).round(6))
print("Per-output residual permutation importance:")
display(importance_df.round(6))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].barh(importance_mean["feature"], importance_mean["importance"], color="steelblue")
axes[0].set_xlabel("Mean permutation importance")
axes[0].set_title("Residual ML — mean feature importance")
axes[0].grid(True, axis="x", alpha=0.3)

im = axes[1].imshow(importance_df.to_numpy(), aspect="auto", cmap="Blues")
axes[1].set_xticks(range(len(INPUT_COLS)))
axes[1].set_xticklabels(INPUT_COLS, rotation=30, ha="right")
axes[1].set_yticks(range(len(OUTPUT_COLS)))
axes[1].set_yticklabels(OUTPUT_COLS, fontsize=8)
axes[1].set_title("Residual ML — importance by output")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

## 9. Hybrid prediction: `ŷ = y_phys + r̂`

Add physics baseline and predicted residual, then clip to physical limits.
If physics fails for a row, that hybrid prediction is left as NaN (validation uses finite rows only).

In [ ]:
def clip_hybrid(y, P_in_bar_g):
    out = y.copy()
    out[0:5] = np.clip(out[0:5], 0.0, 800.0)
    out[5] = min(float(out[5]), float(P_in_bar_g) - 1e-6)
    out[7] = np.clip(out[7], 0.0, 100.0)
    return out


def combine_residual(Y_phys, R_hat):
    """ŷ = y_phys + r̂  (element-wise)."""
    return Y_phys + R_hat


Y_hyb_train = np.where(
    np.isfinite(Y_phys_train),
    combine_residual(Y_phys_train, R_hat_train),
    np.nan,
)
Y_hyb_test = np.where(
    np.isfinite(Y_phys_test),
    combine_residual(Y_phys_test, R_hat_test),
    np.nan,
)

for i in range(len(U_test)):
    if np.isfinite(Y_hyb_test[i]).all():
        Y_hyb_test[i] = clip_hybrid(Y_hyb_test[i], P_in_bar_g=U_test[i, 1])

for i in range(len(U_train)):
    if np.isfinite(Y_hyb_train[i]).all():
        Y_hyb_train[i] = clip_hybrid(Y_hyb_train[i], P_in_bar_g=U_train[i, 1])

print("Hybrid (physics + residual) predictions ready.")
print(f"Finite hybrid test rows: {np.isfinite(Y_hyb_test).all(axis=1).sum()}/{len(Y_hyb_test)}")

## 10. Validate: physics-only vs residual hybrid (test set)

In [ ]:
ok_eval = np.isfinite(Y_phys_test).all(axis=1) & np.isfinite(Y_hyb_test).all(axis=1)

rows = []
for i, col in enumerate(OUTPUT_COLS):
    y_true = Y_excel_test[ok_eval, i]
    y_p = Y_phys_test[ok_eval, i]
    y_h = Y_hyb_test[ok_eval, i]
    rows.append(
        {
            "Output": col,
            "MAE physics": mean_absolute_error(y_true, y_p),
            "MAE residual hybrid": mean_absolute_error(y_true, y_h),
            "R2 physics": r2_score(y_true, y_p),
            "R2 residual hybrid": r2_score(y_true, y_h),
            "MAE of residual model": mean_absolute_error(
                R_test[ok_eval, i], R_hat_test[ok_eval, i]
            ),
        }
    )

metrics = pd.DataFrame(rows)
display(metrics.round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(Y_excel_test[ok_eval, 7], Y_phys_test[ok_eval, 7], alpha=0.4, s=16, label="Physics")
axes[0].scatter(Y_excel_test[ok_eval, 7], Y_hyb_test[ok_eval, 7], alpha=0.5, s=16, label="Phys + residual")
lim = max(Y_excel_test[ok_eval, 7].max(), Y_hyb_test[ok_eval, 7].max(), 1e-6) * 1.1
axes[0].plot([0, lim], [0, lim], "k--", lw=1)
axes[0].set_xlabel("Excel conversion (%)")
axes[0].set_ylabel("Model conversion (%)")
axes[0].set_title("Overall conversion")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(Y_excel_test[ok_eval, 4], Y_phys_test[ok_eval, 4], alpha=0.4, s=16, label="Physics")
axes[1].scatter(Y_excel_test[ok_eval, 4], Y_hyb_test[ok_eval, 4], alpha=0.5, s=16, label="Phys + residual")
t0 = min(Y_excel_test[ok_eval, 4].min(), Y_hyb_test[ok_eval, 4].min()) - 5
t1 = max(Y_excel_test[ok_eval, 4].max(), Y_hyb_test[ok_eval, 4].max()) + 5
axes[1].plot([t0, t1], [t0, t1], "k--", lw=1)
axes[1].set_xlabel("Excel Temp_1m (°C)")
axes[1].set_ylabel("Model Temp_1m (°C)")
axes[1].set_title("Outlet temperature")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle("Residual hybrid twin vs Excel (test set)")
fig.tight_layout()
plt.show()

## 11. Digital twin function

`predict_outputs(T_in, P_in, T_heat, F_in)` runs physics baseline, predicts residual from `u` only, then adds them.

In [ ]:
def predict_outputs(T_in_C, P_in_bar_g, T_heat_C, F_in_kgmole_h):
    """
    Residual hybrid digital twin (Option A):

      y_phys = ODE(η)
      r_hat  = residual_ML(u)     # u only
      y_hat  = clip(y_phys + r_hat)
    """
    phys = predict_physics(T_in_C, P_in_bar_g, T_heat_C, F_in_kgmole_h, eta=ETA)
    y_phys = np.array([phys[c] for c in OUTPUT_COLS], dtype=float)

    u = np.array([[T_in_C, P_in_bar_g, T_heat_C, F_in_kgmole_h]], dtype=float)
    r_hat = residual_model.predict(u)[0]

    y_hat = clip_hybrid(y_phys + r_hat, P_in_bar_g=P_in_bar_g)

    out = {c: float(y_hat[i]) for i, c in enumerate(OUTPUT_COLS)}
    out["y_phys"] = {c: float(y_phys[i]) for i, c in enumerate(OUTPUT_COLS)}
    out["r_hat"] = {c: float(r_hat[i]) for i, c in enumerate(OUTPUT_COLS)}
    out["z_m"] = phys["z_m"]
    out["T_profile_C"] = phys["T_profile_C"]
    return out


demo = df.iloc[0]
demo_pred = predict_outputs(demo["T_in_C"], demo["P_in_bar_g"], demo["T_heat_C"], demo["F_in_kgmole_h"])

smoke = pd.DataFrame(
    {
        "Output": OUTPUT_COLS,
        "Excel": [demo[c] for c in OUTPUT_COLS],
        "Physics": [demo_pred["y_phys"][c] for c in OUTPUT_COLS],
        "Residual r̂": [demo_pred["r_hat"][c] for c in OUTPUT_COLS],
        "Hybrid": [demo_pred[c] for c in OUTPUT_COLS],
    }
)
smoke["Abs error"] = (smoke["Hybrid"] - smoke["Excel"]).abs()
print("Smoke test — first cleaned Methanol Reforming Data.xlsx row:")
display(smoke.round(4))


---

## 12. Interactive digital twin

Enter inlet conditions → physics baseline + ML residual correction.

In [ ]:
def _ask_float(prompt, default=None):
    hint = f" [{default}]" if default is not None else ""
    while True:
        raw = input(f"{prompt}{hint}: ").strip()
        if raw.lower() in {"q", "quit", "exit"}:
            return None
        if raw == "" and default is not None:
            return float(default)
        try:
            return float(raw)
        except ValueError:
            print("  Please enter a number (or q to quit).")


def run_digital_twin_interactive():
    print("\n" + "=" * 62)
    print("  METHANOL REACTOR — RESIDUAL HYBRID DIGITAL TWIN")
    print("  ŷ = y_phys + r̂(u)     (Option A: ML sees inputs only)")
    print("=" * 62)
    print(f"  Physics η = {ETA:.6e}")
    print("  Press Enter for defaults. Type q to stop.\n")

    while True:
        T_in_C = _ask_float("Inlet temperature [°C]", default=float(df["T_in_C"].iloc[0]))
        if T_in_C is None:
            print("Digital twin stopped.")
            break

        P_in_bar_g = _ask_float("Inlet pressure [bar_g]", default=float(df["P_in_bar_g"].iloc[0]))
        T_heat_C = _ask_float(
            "Heating media temperature [C]",
            default=float(df["T_heat_C"].iloc[0]),  # Excel first-case default
        )
        if P_in_bar_g is None:
            print("Digital twin stopped.")
            break

        F_in_kgmole_h = _ask_float("Methanol molar flow [kgmole/h]", default=float(df["F_in_kgmole_h"].iloc[0]))
        if F_in_kgmole_h is None:
            print("Digital twin stopped.")
            break

        if F_in_kgmole_h <= 0:
            print("  Flow must be > 0. Try again.\n")
            continue

        try:
            pred = predict_outputs(T_in_C, P_in_bar_g, T_heat_C, F_in_kgmole_h)
        except Exception as exc:
            print(f"  Prediction failed: {exc}\n")
            continue

        print("\n" + "-" * 62)
        print("INPUTS")
        print(f"  Inlet Temperature     : {T_in_C:.2f} °C")
        print(f"  Inlet Pressure        : {P_in_bar_g:.4f} bar_g")
        print(f"  Methanol flow         : {F_in_kgmole_h:.4f} kgmole/h")
        print("\nHYBRID OUTPUTS (physics + residual)")
        print(f"  Temp_0.2m             : {pred['T_0.2m_C']:.2f} °C")
        print(f"  Temp_0.4m             : {pred['T_0.4m_C']:.2f} °C")
        print(f"  Temp_0.6m             : {pred['T_0.6m_C']:.2f} °C")
        print(f"  Temp_0.8m             : {pred['T_0.8m_C']:.2f} °C")
        print(f"  Temp_1m (outlet)      : {pred['T_1m_C']:.2f} °C")
        print(f"  Outlet Pressure       : {pred['P_out_bar_g']:.4f} bar_g")
        print(f"  Total heat input      : {pred['Q_total_kW']:.4f} kW")
        print(f"  Overall Conversion    : {pred['X_overall_pct']:.4f} %")
        print("-" * 62)

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(pred["z_m"], pred["T_profile_C"], lw=2, alpha=0.7, label="Physics T(z)")
        ax.scatter(
            MEASUREMENT_POSITIONS_M,
            [pred[c] for c in TEMP_COLS],
            color="crimson",
            zorder=5,
            label="Hybrid T (phys + residual)",
        )
        ax.axhline(T_heat_C, color="gray", ls="--", alpha=0.6, label=f"Heat medium ({T_heat_C:.0f} °C)")
        ax.set_xlabel("Reactor length (m)")
        ax.set_ylabel("Temperature (°C)")
        ax.set_title("Residual hybrid twin — temperatures")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        again = input("\nRun another case? [Y/n]: ").strip().lower()
        if again in {"n", "no", "q", "quit"}:
            print("Digital twin stopped.")
            break


run_digital_twin_interactive()